<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Appendix H: *Reservoir Processing*

## Metadata

---
### Contents  
> 1. ** 
> 2. **
---
### Notes

- 
---
### Inputs
- 
---
### Outputs  
- 
---
### User Created Dependencies  
---

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
import pandas as pd
import geopandas as gpd

In [3]:
res1 = pd.read_csv('../data/raw/res1.csv')
res2 = pd.read_csv('../data/raw/res2.csv')
res3 = pd.read_csv('../data/raw/res3.csv')

In [4]:
stations = pd.read_csv('../data/raw/Reservoir_stations2.csv')

In [5]:
res = pd.concat([res1,res2,res3])

In [6]:
res['UNITS'].value_counts()

UNITS
AF    452994
Name: count, dtype: int64

In [7]:
res = res.rename(columns = {
    'STATION_ID': 'Station_ID',
    'DATE TIME' : 'Date',
    'VALUE': 'Reservoir Level'
})

res = res.drop(columns=['DURATION','SENSOR_NUMBER','SENS_TYPE','OBS DATE','DATA_FLAG','UNITS'])

In [8]:
res['Date'] = pd.to_datetime(res['Date'])

In [9]:
res['Reservoir Level'] = pd.to_numeric(res['Reservoir Level'].str.replace(',', ''), errors='coerce')

In [10]:
stations = stations.drop(columns=['Station','County','Agency'])

In [11]:
stations

,Station_ID,Elevation,Latitude,Longitude
0,SWV,25,38.582001,-121.500000
1,DRE,2805,41.540894,-122.374550
2,CLE,2370,40.800999,-122.762001
3,LEW,1870,40.727001,-122.792999
4,RTD,2675,40.367001,-123.432999
...,...,...,...,...
204,TNM,3882,37.057999,-118.224998
205,HWE,3774,36.137001,-117.947998
206,SKR,837,37.166019,-118.569542
207,LRK,745,34.485001,-118.022003


In [12]:
res_merged = pd.merge(
    left=res,
    right = stations,
    on = 'Station_ID',
    how = 'inner'
)

In [13]:
sort_res_merged = res_merged.sort_values('Date')

In [14]:
sort_res_merged["Reservoir Level_has_data"] = sort_res_merged["Reservoir Level"].notna().astype(int)
sort_res_merged['Reservoir Level'] = sort_res_merged['Reservoir Level'].fillna(0)

In [15]:
sort_res_merged.to_csv('../data/raw/reservoirs.csv',index = False)
print("All datasets saved successfully")